In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session


get_ipython().getoutput("pip install -q segmentation-models-pytorch albumentations")


import os
import cv2
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import segmentation_models_pytorch as smp
import albumentations as A
from albumentations.pytorch import ToTensorV2
import matplotlib.pyplot as plt
from tqdm import tqdm


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using:", device)


BASE_PATH = "/kaggle/input/datasets/balraj98/massachusetts-buildings-dataset"

TRAIN_IMG_PATH = os.path.join(BASE_PATH, "tiff/train")
TRAIN_MASK_PATH = os.path.join(BASE_PATH, "tiff/train_labels")

VAL_IMG_PATH = os.path.join(BASE_PATH, "tiff/val")
VAL_MASK_PATH = os.path.join(BASE_PATH, "tiff/val_labels")


train_transform = A.Compose([
    A.HorizontalFlip(p=0.5),
    A.VerticalFlip(p=0.5),
    A.RandomRotate90(p=0.5),
    A.RandomBrightnessContrast(p=0.3),
    A.Normalize(mean=(0.485, 0.456, 0.406),
                std=(0.229, 0.224, 0.225)),
    ToTensorV2()
])

val_transform = A.Compose([
    A.Normalize(mean=(0.485, 0.456, 0.406),
                std=(0.229, 0.224, 0.225)),
    ToTensorV2()
])


def extract_patches(img, mask, patch_size=256):
    img_patches = []
    mask_patches = []

    h, w = img.shape[:2]

    for i in range(0, h - patch_size + 1, patch_size):
        for j in range(0, w - patch_size + 1, patch_size):
            img_patch = img[i:i+patch_size, j:j+patch_size]
            mask_patch = mask[i:i+patch_size, j:j+patch_size]

            img_patches.append(img_patch)
            mask_patches.append(mask_patch)

    return img_patches, mask_patches


class BuildingDataset(Dataset):
    def __init__(self, img_dir, mask_dir, transform=None, patch_size=256):
        self.transform = transform
        self.patch_size = patch_size

        self.img_patches = []
        self.mask_patches = []

        images = sorted(os.listdir(img_dir))
        masks = sorted(os.listdir(mask_dir))

        for img_name, mask_name in zip(images, masks):
            img = cv2.imread(os.path.join(img_dir, img_name))
            img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

            mask = cv2.imread(os.path.join(mask_dir, mask_name), 0)
            mask = (mask > 0).astype(np.float32)

            img_p, mask_p = extract_patches(img, mask, self.patch_size)

            self.img_patches.extend(img_p)
            self.mask_patches.extend(mask_p)

    def __len__(self):
        return len(self.img_patches)

    def __getitem__(self, idx):
        img = self.img_patches[idx]
        mask = self.mask_patches[idx]

        if self.transform:
            augmented = self.transform(image=img, mask=mask)
            img = augmented["image"]
            mask = augmented["mask"].unsqueeze(0)
        else:
            img = torch.tensor(img).permute(2,0,1).float() / 255.0
            mask = torch.tensor(mask).unsqueeze(0)

        return img, mask.float()


train_dataset = BuildingDataset(
    TRAIN_IMG_PATH,
    TRAIN_MASK_PATH,
    transform=train_transform,
    patch_size=256   
)

val_dataset = BuildingDataset(
    VAL_IMG_PATH,
    VAL_MASK_PATH,
    transform=val_transform,
    patch_size=256  
)


train_dataset = BuildingDataset(TRAIN_IMG_PATH, TRAIN_MASK_PATH, patch_size=256)
val_dataset = BuildingDataset(VAL_IMG_PATH, VAL_MASK_PATH, patch_size=256)

train_loader = DataLoader(train_dataset, batch_size=8, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=8)


model = smp.Unet(
    encoder_name="efficientnet-b3",
    encoder_weights="imagenet",
    in_channels=3,
    classes=1,
    activation=None
)

model.to(device)


loss_fn = smp.losses.DiceLoss(mode='binary')
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)


loss_fn = smp.losses.DiceLoss(mode='binary', from_logits=True)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)

scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='max', patience=3, factor=0.5
)


def iou_score(pred, mask):
    pred = torch.sigmoid(pred)
    pred = (pred > 0.5).float()
    intersection = (pred * mask).sum()
    union = pred.sum() + mask.sum() - intersection
    return (intersection + 1e-6) / (union + 1e-6)

epochs = 30

best_iou = 0
train_losses = []
val_losses = []
ious = []

for epoch in range(epochs):

    # ---- TRAIN ----
    model.train()
    train_loss = 0

    for imgs, masks in tqdm(train_loader):
        imgs = imgs.to(device)
        masks = masks.to(device)

        preds = model(imgs)
        loss = loss_fn(preds, masks)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        train_loss += loss.item()

    avg_train_loss = train_loss / len(train_loader)
    train_losses.append(avg_train_loss)

    # ---- VALIDATION ----
    model.eval()
    val_loss = 0
    iou_total = 0

    with torch.no_grad():
        for imgs, masks in val_loader:
            imgs = imgs.to(device)
            masks = masks.to(device)

            preds = model(imgs)
            loss = loss_fn(preds, masks)

            val_loss += loss.item()
            iou_total += iou_score(preds, masks).item()

    avg_val_loss = val_loss / len(val_loader)
    avg_iou = iou_total / len(val_loader)

    val_losses.append(avg_val_loss)
    ious.append(avg_iou)

    scheduler.step(avg_iou)

    print(f"\nEpoch {epoch+1}")
    print(f"Train Loss: {avg_train_loss:.4f}")
    print(f"Val Loss: {avg_val_loss:.4f}")
    print(f"Val IoU: {avg_iou:.4f}")

    if avg_iou > best_iou:
        best_iou = avg_iou
        torch.save(model.state_dict(), "best_model.pth")
        print("Best model saved!")


plt.figure(figsize=(12,5))

plt.subplot(1,2,1)
plt.plot(train_losses, label="Train")
plt.plot(val_losses, label="Val")
plt.title("Loss Curve")
plt.legend()

plt.subplot(1,2,2)
plt.plot(ious)
plt.title("IoU Curve")

plt.show()


model.load_state_dict(torch.load("best_model.pth"))
model.eval()

imgs, masks = next(iter(val_loader))
imgs = imgs.to(device)

with torch.no_grad():
    preds = model(imgs)

pred = torch.sigmoid(preds[0]).cpu().numpy().squeeze()
pred = (pred > 0.5)

img = imgs[0].cpu().permute(1,2,0).numpy()

plt.figure(figsize=(10,4))

plt.subplot(1,2,1)
plt.imshow(img)
plt.title("Input")

plt.subplot(1,2,2)
plt.imshow(pred, cmap='gray')
plt.title("Prediction")

plt.show()


In [1]:
import cv2
import numpy as np


def create_zoning_mask(shape):
    h, w = shape
    zoning = np.zeros((h, w), dtype=np.uint8)
    zoning[:, w//2:] = 1
    return zoning


def get_building_components(binary_mask):
    num_labels, labels = cv2.connectedComponents(binary_mask.astype(np.uint8))
    return num_labels, labels


def detect_illegal_buildings(building_mask, zoning_mask):

    num_labels, labels = get_building_components(building_mask)

    illegal_buildings = []
    legal_buildings = []

    for label in range(1, num_labels):  # skip background (0)
        building_pixels = (labels == label)

        # Check overlap with restricted zone
        overlap = building_pixels & (zoning_mask == 1)

        if overlap.any():
            illegal_buildings.append(label)
        else:
            legal_buildings.append(label)

    return illegal_buildings, legal_buildings, labels


def visualize_illegal(image, labels, illegal_buildings):

    output = image.copy()

    for label in illegal_buildings:
        output[labels == label] = [255, 0, 0]  # red

    return output


plt.figure(figsize=(12,4))

plt.subplot(1,3,1)
plt.title("Building Mask")
plt.imshow(pred_mask, cmap='gray')

plt.subplot(1,3,2)
plt.title("Zoning Mask")
plt.imshow(zoning_mask, cmap='gray')

plt.subplot(1,3,3)
plt.title("Overlay Result")
plt.imshow(overlay)

plt.show()


pred_mask = (pred > 0.5).astype(np.uint8)

zoning_mask = create_zoning_mask(pred_mask.shape)

illegal_buildings, legal_buildings, labels = detect_illegal_buildings(
    pred_mask,
    zoning_mask
)

overlay = visualize_illegal(img.astype(np.uint8), labels, illegal_buildings)

print("Total Buildings:", len(illegal_buildings) + len(legal_buildings))
print("Illegal Buildings:", len(illegal_buildings))
print("Legal Buildings:", len(legal_buildings))

plt.figure(figsize=(8,6))
plt.imshow(overlay)
plt.title("Illegal Buildings Highlighted in Red")
plt.show()


get_ipython().getoutput("ls /kaggle/working")


ModuleNotFoundError: No module named 'cv2'